# Zepto Data Pipeline: Web Scraping to Relational SQLite Store

This notebook demonstrates the end-to-end catalog data engineering pipeline for Zepto:
1. **Scraping**: Extracting product listings from `http://books.toscrape.com/` (100 books across 29 categories).
2. **Cleaning & Transformation**: Parsing raw prices, word-to-number star ratings, and stock status.
3. **Currency Conversion**: Fixed baseline conversion ($1\text{ GBP} = 105.50\text{ INR}$).
4. **Normalized SQLite Storage**: Two-table schema (`categories` and `books`) with Primary Key / Foreign Key constraints.
5. **Analytical SQL Queries**: Covering `WHERE`, `ORDER BY`, `LIMIT`, `DISTINCT`, `IN/BETWEEN`, and `JOIN` with aggregations.
6. **Pandas Equivalence Verification**: Proving that in-memory `pd.merge()` yields the exact same result as the SQL `JOIN`.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup

from scraper import scrape_books_catalog, clean_and_transform_data, FIXED_GBP_TO_INR_RATE
from pipeline import init_database, populate_database, execute_analytical_queries, verify_pandas_merge_equivalence, DB_PATH

print("Modules imported successfully.")

Modules imported successfully.


## 1. Web Scraping & Raw Data Extraction
We scrape the first 5 paginated catalog pages from `http://books.toscrape.com/` to retrieve 100 books across diverse categories.

In [2]:
raw_records = scrape_books_catalog(max_pages=5)
print(f"Total raw records scraped: {len(raw_records)}")
pd.DataFrame(raw_records).head()

2026-09-20 22:13:06,078 - INFO - Starting scraping from http://books.toscrape.com/ across 5 pages...


2026-09-20 22:13:08,928 - INFO - Found 50 categories. Scraping catalog pages...


2026-09-20 22:13:08,930 - INFO - Scraping catalog page 1: http://books.toscrape.com/catalogue/page-1.html


2026-09-20 22:13:33,122 - INFO - Scraping catalog page 2: http://books.toscrape.com/catalogue/page-2.html


2026-09-20 22:14:00,574 - INFO - Scraping catalog page 3: http://books.toscrape.com/catalogue/page-3.html


2026-09-20 22:14:28,285 - INFO - Scraping catalog page 4: http://books.toscrape.com/catalogue/page-4.html


2026-09-20 22:15:04,263 - INFO - Scraping catalog page 5: http://books.toscrape.com/catalogue/page-5.html


2026-09-20 22:15:50,610 - INFO - Scraped total 100 books across multiple categories.


Total raw records scraped: 100


,title,price_raw,star_rating_raw,availability_raw,category
0,A Light in the Attic,£51.77,Three,In stock,Poetry
1,Tipping the Velvet,£53.74,One,In stock,Historical Fiction
2,Soumission,£50.10,One,In stock,Fiction
3,Sharp Objects,£47.82,Four,In stock,Mystery
4,Sapiens: A Brief History of Humankind,£54.23,Five,In stock,History


## 2. Data Cleaning, Type Casting & Fixed Currency Conversion
- Strip currency symbols -> `price_gbp` (`float64`)
- Star ratings ('One'...'Five') -> `rating` (`int` 1-5)
- Stock availability -> `in_stock` (`bool`)
- Median imputation for numeric errors
- Currency conversion using fixed rate: `1 GBP = 105.50 INR` -> `price_inr`

In [3]:
clean_df = clean_and_transform_data(raw_records)
print("Cleaned Data Info:")
print(clean_df.info())
clean_df.head(10)

2026-09-20 22:15:51,362 - INFO - Cleaning 100 scraped records...


2026-09-20 22:15:51,659 - INFO - Data cleaning completed successfully. Resulting shape: (100, 6)


2026-09-20 22:15:51,777 - INFO - Cleaned column types:
title            str
category         str
price_gbp    float64
price_inr    float64
rating         int64
in_stock        bool
dtype: object


Cleaned Data Info:


<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      100 non-null    str    
 1   category   100 non-null    str    
 2   price_gbp  100 non-null    float64
 3   price_inr  100 non-null    float64
 4   rating     100 non-null    int64  
 5   in_stock   100 non-null    bool   
dtypes: bool(1), float64(2), int64(1), str(2)
memory usage: 4.1 KB
None


,title,category,price_gbp,price_inr,rating,in_stock
0,A Light in the Attic,Poetry,51.77,5461.74,3,True
1,Tipping the Velvet,Historical Fiction,53.74,5669.57,1,True
2,Soumission,Fiction,50.10,5285.55,1,True
3,Sharp Objects,Mystery,47.82,5045.01,4,True
4,Sapiens: A Brief History of Humankind,History,54.23,5721.26,5,True
5,The Requiem Red,Young Adult,22.65,2389.57,1,True
6,The Dirty Little Secrets of Getting Your Dream...,Business,33.34,3517.37,4,True
7,The Coming Woman: A Novel Based on the Life of...,Default,17.93,1891.62,3,True
8,The Boys in the Boat: Nine Americans and Their...,Default,22.60,2384.30,4,True
9,The Black Maria,Poetry,52.15,5501.82,1,True


## 3. Normalized Relational Database Loading (`books.db`)
Creating normalized tables `categories` and `books` with Foreign Key constraints.

In [4]:
conn = init_database(DB_PATH)
populate_database(clean_df, conn)
print("SQLite database initialized and populated at:", DB_PATH)

2026-09-20 22:15:53,103 - INFO - Initialized normalized SQLite schema at C:\Users\aduru\OneDrive\Desktop\Masai capstone project\data_pipeline\books.db


2026-09-20 22:15:53,704 - INFO - Loaded 29 categories and 100 books into SQLite.


SQLite database initialized and populated at: C:\Users\aduru\OneDrive\Desktop\Masai capstone project\data_pipeline\books.db


## 4. Analytical SQL Queries
Executing 6 analytical queries demonstrating filtering, sorting, limiting, distinct values, ranges, and relational joins.

In [5]:
results = execute_analytical_queries(conn)

for qkey, (desc, sql, df_res) in results.items():
    print("=" * 80)
    print(f"[{qkey}] {desc}")
    print("SQL Query:\n" + sql)
    print(f"Returned {len(df_res)} rows:")
    display(df_res.head(5))

2026-09-20 22:15:54,138 - INFO - Executed Q1_SELECT_WHERE (Query 1: Filter high-rated books with price above INR 3,000 (SELECT / WHERE)) -> 19 rows returned.


2026-09-20 22:15:54,289 - INFO - Executed Q2_ORDER_BY_LIMIT (Query 2: Top 5 most expensive books in stock (ORDER BY / LIMIT)) -> 5 rows returned.


2026-09-20 22:15:54,458 - INFO - Executed Q3_DISTINCT (Query 3: Distinct categories available in catalog (DISTINCT)) -> 29 rows returned.


2026-09-20 22:15:54,571 - INFO - Executed Q4_IN_BETWEEN (Query 4: Books with rating IN (4, 5) and price_gbp BETWEEN 20 and 45 (IN / BETWEEN)) -> 19 rows returned.


2026-09-20 22:15:54,690 - INFO - Executed Q5_JOIN_AGGREGATION (Query 5: Category catalog summary metrics (JOIN / GROUP BY / Aggregations)) -> 29 rows returned.


...

,book_id,title,rating,price_gbp,price_inr
0,66,The Inefficiency Assassin: Time Management Tac...,5,20.59,2172.24
1,44,#HigherSelfie: Wake Up Your Life. Free Your So...,5,23.11,2438.10
2,33,The Elephant Tree,5,23.82,2513.01
3,24,Chase Me (Paris Nights #2),5,25.27,2665.98
4,73,The Activist's Tao Te Ching: Ancient Advice fo...,5,32.24,3401.32


[Q5_JOIN_AGGREGATION] Query 5: Category catalog summary metrics (JOIN / GROUP BY / Aggregations)
SQL Query:
SELECT 
                c.category_name,
                COUNT(b.book_id) AS total_books,
                ROUND(AVG(b.price_inr), 2) AS avg_price_inr,
                ROUND(MIN(b.price_inr), 2) AS min_price_inr,
                ROUND(MAX(b.price_inr), 2) AS max_price_inr,
                MAX(b.rating) AS top_rating
            FROM categories c
            JOIN books b ON c.category_id = b.category_id
            GROUP BY c.category_name
            ORDER BY total_books DESC, avg_price_inr DESC;
Returned 29 rows:


,category_name,total_books,avg_price_inr,min_price_inr,max_price_inr,top_rating
0,Sequential Art,14,3366.13,1071.88,5516.60,5
1,Nonfiction,12,3426.90,1769.24,5914.33,5
2,Default,9,2832.44,1475.94,5605.22,5
3,Poetry,7,3823.17,1505.48,6046.20,4
4,Fiction,5,4628.49,1821.98,5708.60,5


[Q6_JOIN_DETAILED] Query 6: Top 10 highest-rated books with their Category Name (JOIN / ORDER BY / LIMIT)
SQL Query:
SELECT 
                b.book_id,
                b.title,
                c.category_name,
                b.rating,
                b.price_gbp,
                b.price_inr
            FROM books b
            JOIN categories c ON b.category_id = c.category_id
            WHERE b.in_stock = 1
            ORDER BY b.rating DESC, b.price_gbp DESC
            LIMIT 10;
Returned 10 rows:


,book_id,title,category_name,rating,price_gbp,price_inr
0,5,Sapiens: A Brief History of Humankind,History,5,54.23,5721.26
1,14,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,5,52.29,5516.60
2,47,"We Love You, Charlie Freeman",Fiction,5,50.27,5303.48
3,43,Private Paris (Private #10),Fiction,5,47.61,5022.85
4,29,Worlds Elsewhere: Journeys Around Shakespeare’...,Nonfiction,5,40.30,4251.65


## 5. SQL JOIN vs. In-Memory Pandas Merge Equivalence
Comparing `pd.read_sql` JOIN output against in-memory `pd.merge()` without SQL to prove data pipeline correctness.

In [6]:
sql_df, pd_df, is_eq = verify_pandas_merge_equivalence(conn)

print("SQL JOIN Output (pd.read_sql):")
display(sql_df)

print("\nPandas Merge Output (pd.merge):")
display(pd_df)

print(f"\nExact Match Verified: {is_eq}")
assert is_eq, "Equivalence check failed!"
print("Assertion Passed: Both approaches produce identical outputs.")

conn.close()

2026-09-20 22:15:58,798 - INFO - Pandas vs SQL JOIN equivalence check: PASSED (Exact Match)


SQL JOIN Output (pd.read_sql):


,book_id,title,category_name,rating,price_gbp,price_inr
0,5,Sapiens: A Brief History of Humankind,History,5,54.23,5721.26
1,14,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,5,52.29,5516.60
2,47,"We Love You, Charlie Freeman",Fiction,5,50.27,5303.48
3,43,Private Paris (Private #10),Fiction,5,47.61,5022.85
4,29,Worlds Elsewhere: Journeys Around Shakespeare’...,Nonfiction,5,40.30,4251.65
5,99,Join,Science Fiction,5,35.67,3763.19
6,15,Rip it Up and Start Again,Music,5,35.02,3694.61
7,25,Black Dust,Romance,5,34.53,3642.92
8,73,The Activist's Tao Te Ching: Ancient Advice fo...,Spirituality,5,32.24,3401.32
9,24,Chase Me (Paris Nights #2),Romance,5,25.27,2665.98



Pandas Merge Output (pd.merge):


,book_id,title,category_name,rating,price_gbp,price_inr
0,5,Sapiens: A Brief History of Humankind,History,5,54.23,5721.26
1,14,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,5,52.29,5516.60
2,47,"We Love You, Charlie Freeman",Fiction,5,50.27,5303.48
3,43,Private Paris (Private #10),Fiction,5,47.61,5022.85
4,29,Worlds Elsewhere: Journeys Around Shakespeare’...,Nonfiction,5,40.30,4251.65
5,99,Join,Science Fiction,5,35.67,3763.19
6,15,Rip it Up and Start Again,Music,5,35.02,3694.61
7,25,Black Dust,Romance,5,34.53,3642.92
8,73,The Activist's Tao Te Ching: Ancient Advice fo...,Spirituality,5,32.24,3401.32
9,24,Chase Me (Paris Nights #2),Romance,5,25.27,2665.98



Exact Match Verified: True


Assertion Passed: Both approaches produce identical outputs.
